# Chains
- A sequence of components where the output of one step becomes the input of the next
- Think of it like: User Input → Prompt → LLM → Output Parser → Final Answer

### Why Chains ?
1. Without chains:
    - You manually manage prompts, outputs, parsing
    - Hard to scale logic
    - No modularity

2. With chains:
    - Reusable pipelines
    - Clean abstraction
    - Easier debugging & extension
    - Production-ready workflows (important for MLOps)

### There are commonly 4 types of chains:
1. Simple Chains
2. Sequential Chains
3. Parallel Chains
4. Conditional Chains

### Building Blocks of a chain:
1. Prompt
2. LLM/ Chat Model
3. Output Parsers
4. Memory (Optional)
5. Tools/ Retrievers (Advanced)  

Using chains in your application, this method is also known as LCEL - Langchain Expression Language

In [4]:
# Simple Chain
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

model = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
parser = StrOutputParser()

chain = prompt | model | parser
result = chain.invoke({'topic':'cricket'})

print("Result:", result)
print("\n---------------------Graph:-------------------\n")
chain.get_graph().print_ascii()

Result: **5 Fascinating Cricket Facts**

1. **The Longest Match in History**  
   The 1939–40 Ashes Test between England and Australia at Adelaide lasted 10 days (including a 2‑day break for a storm). It was the longest Test ever played, with 1,000+ overs bowled and 1,000+ runs scored.

2. **The “Century” Origin**  
   A batsman scoring 100 runs in a single innings is called a “century.” The term comes from the 19th‑century English cricket tradition of awarding a silver cup to the first player to reach 100 runs in a match.

3. **The Fastest Delivery in International Cricket**  
   In 2011, Australian bowler Shaun Tait bowled a ball at 161.3 km/h (100.23 mph) against England. It remains the fastest recorded delivery in a One‑Day International (ODI) match.

4. **The “Mankading” Rule**  
   Named after Indian bowler Vinoo Mankad, the rule allows a bowler to run out a non‑striker who leaves the crease early. Though controversial, it’s a legitimate dismissal under the Laws of Cricket.

5. *

In [5]:
# Sequential Chain
from dotenv import load_dotenv
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

parser = StrOutputParser()
chain = prompt1 | model | parser | prompt2 | model | parser
result = chain.invoke({'topic': 'Unemployment in India'})

print("Result:", result)
print("\n---------------------Graph:-------------------\n")
chain.get_graph().print_ascii()

Result: **5‑Point Summary of the Unemployment Report (India, 2024–2026)**  

1. **National Trend & Key Figures** – The overall unemployment rate has eased to **6.8 %** (2024), with youth (15‑29 yrs) at **13.2 %** and female unemployment at **9.1 %**, both markedly above the national average.  

2. **Sectoral & Informal‑Economy Dynamics** – While **70 %** of jobs are in the informal sector, it also accounts for **65 %** of unemployment, underscoring a mismatch between job creation and job quality.  

3. **Regional Disparities** – States with robust industrial/IT ecosystems (Gujarat, Maharashtra, Karnataka) maintain rates below 5 %, whereas agrarian and less developed regions (Bihar, UP, Rajasthan, North‑East) face unemployment above 10 %.  

4. **Primary Drivers** – Rapid population growth, skill mismatch, slow formal‑sector expansion, lingering COVID‑19 effects, and technological displacement are the main forces sustaining unemployment.  

5. **Policy Outlook & Recommendations** – Exis

In [6]:
# Parallel Chain
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)
model2 = ChatGroq(model="openai/gpt-oss-safeguard-20b", temperature=0)

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser

chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text':text})

print("Result:", result)
print("\n---------------------Graph:-------------------\n")
chain.get_graph().print_ascii()


Result: # Support Vector Machines (SVMs)

## Overview
Support Vector Machines (SVMs) are a versatile supervised learning tool used for:
- **Classification**  
- **Regression**  
- **Outlier detection**

They are particularly powerful in high‑dimensional spaces and can handle situations where the number of features exceeds the number of samples.

---

## Key Advantages

| Feature | Description |
|---------|-------------|
| **High‑dimensional performance** | Works well even when dimensions > samples. |
| **Memory efficiency** | Uses only a subset of training points – the *support vectors* – in the decision function. |
| **Flexibility** | Supports built‑in or custom kernel functions to shape the decision boundary. |
| **Scikit‑learn compatibility** | Accepts both dense (`numpy.ndarray`) and sparse (`scipy.sparse`) inputs. |
| **Optimal data formats** | Best performance with `C`‑ordered `numpy.ndarray` (dense) or `scipy.sparse.csr_matrix` (sparse) using `dtype=float64`. |

---

## Main Dis

In [8]:
# Conditional Chain
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv()

class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser = StrOutputParser()
parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print("Result:", chain.invoke({'feedback': 'The phone quality & performance is really poor'}))
print("\n---------------------Graph:-------------------\n")
chain.get_graph().print_ascii()

Result: Thank you for sharing your concerns. I’m sorry to hear that your experience didn’t meet your expectations. Could you please let me know a bit more about what went wrong? Your feedback is invaluable, and I’d like to investigate the issue and find a solution that works for you.

---------------------Graph:-------------------

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
      | ChatGroq |       
      +----------+       
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch 